# Claude Opus 4.7 Cookbook

Practical workflows for `claude-opus-4-7` with the Anthropic Python SDK.

What this notebook is built around:
- `1M` token context window
- `$5 / million input tokens` and `$25 / million output tokens`
- `xhigh` for coding and agentic work
- `thinking={"type": "adaptive"}` when you want adaptive thinking
- structured outputs, streaming, tools, retries, latency tracking, and batch runs

Important migration note: Claude Opus 4.7 rejects non-default `temperature`, `top_p`, and `top_k` values. This notebook uses effort controls instead.

## 1. Install and import dependencies

Install the notebook dependencies once, then verify the versions you will actually run with.

In [ ]:
# Uncomment this line in a fresh environment.
# !pip install anthropic pandas pydantic rich tenacity

import ast
import base64
import json
import os
import time
from pathlib import Path
from typing import Any

import anthropic
import pandas as pd
from anthropic import Anthropic, beta_tool
from pydantic import BaseModel, Field, ValidationError
from rich.console import Console
from rich.table import Table
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential_jitter

print("anthropic", anthropic.__version__)
print("pandas", pd.__version__)
print("pydantic", BaseModel.__module__.split(".")[0])
print("rich", Console.__module__.split(".")[0])
print("tenacity", retry.__module__.split(".")[0])

## 2. Configure API credentials and model constants

`claude-opus-4-7` is the model string to use here. Effort is the main control knob for this model, and `xhigh` is the recommended starting point for coding or agentic work.

In [ ]:
API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not API_KEY:
    raise RuntimeError("Set ANTHROPIC_API_KEY before running live API calls.")

MODEL = "claude-opus-4-7"
DEFAULT_MAX_TOKENS = 4096
DEFAULT_EFFORT = "high"
DEFAULT_THINKING = {"type": "adaptive"}
DEFAULT_TASK_BUDGET_TOKENS = None

client = Anthropic(api_key=API_KEY)

print({
    "model": MODEL,
    "default_max_tokens": DEFAULT_MAX_TOKENS,
    "default_effort": DEFAULT_EFFORT,
    "adaptive_thinking": DEFAULT_THINKING,
    "task_budget": DEFAULT_TASK_BUDGET_TOKENS,
})

## 3. Create a reusable Claude API helper

This helper keeps Opus 4.7-specific settings in one place, including effort, adaptive thinking, and the optional task budget beta path.

In [ ]:
def extract_text(message: Any) -> str:
    blocks = []
    for block in message.content:
        text = getattr(block, "text", None)
        if text:
            blocks.append(text)
    return "".join(blocks)


def call_claude(
    prompt: str,
    *,
    system: str | None = None,
    max_tokens: int = DEFAULT_MAX_TOKENS,
    effort: str = DEFAULT_EFFORT,
    use_adaptive_thinking: bool = False,
    output_schema: dict[str, Any] | None = None,
    task_budget_tokens: int | None = DEFAULT_TASK_BUDGET_TOKENS,
) -> dict[str, Any]:
    request: dict[str, Any] = {
        "model": MODEL,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}],
        "output_config": {"effort": effort},
    }

    if system:
        request["system"] = system
    if use_adaptive_thinking:
        request["thinking"] = {"type": "adaptive"}
    if output_schema is not None:
        request["output_config"]["format"] = {"type": "json_schema", "schema": output_schema}
    if task_budget_tokens is not None:
        request["output_config"]["task_budget"] = {"type": "tokens", "total": task_budget_tokens}
        request["betas"] = ["task-budgets-2026-03-13"]

    response = client.messages.create(**request)
    return {
        "text": extract_text(response),
        "response": response,
        "usage": response.usage,
        "request_id": response._request_id,
        "stop_reason": response.stop_reason,
    }

## 4. Run a minimal Opus 4.7 completion

Start with a short prompt so you can confirm connectivity, model selection, and the default request shape before building anything larger.

In [ ]:
result = call_claude(
    "Explain in three bullet points why Claude Opus 4.7 is a good fit for hard coding tasks.",
    max_tokens=300,
    effort="high",
    use_adaptive_thinking=True,
)

print(result["text"])
print({
    "request_id": result["request_id"],
    "stop_reason": result["stop_reason"],
    "usage": result["usage"].model_dump(),
})

## 5. Apply system prompts and prompt templates

Opus 4.7 is more literal than older Claude models, so small prompt changes matter. Use templates to make that behavior predictable.

In [ ]:
system_prompt = "You are a pragmatic assistant for software teams. Keep answers concrete, short, and actionable."

prompt_variants = {
    "direct": "Summarize the release for a backend engineer.",
    "diagnostic": "Summarize the release for a backend engineer, then list the one migration risk that matters most.",
}

for name, user_prompt in prompt_variants.items():
    output = call_claude(
        f"Claude Opus 4.7 release note:\n\n{user_prompt}",
        system=system_prompt,
        max_tokens=350,
        effort="medium",
    )
    print(f"\n=== {name} ===")
    print(output["text"])

## 6. Stream tokens to notebook output

Use streaming when the model needs room to think or when you want live progress in a long notebook cell. The context manager below is cancellation-safe.

In [ ]:
def stream_completion(prompt: str, *, max_tokens: int = 1024, effort: str = "high") -> Any:
    with client.messages.stream(
        model=MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
        output_config={"effort": effort},
    ) as stream:
        try:
            for chunk in stream.text_stream:
                print(chunk, end="", flush=True)
            print()
            return stream.get_final_message()
        except KeyboardInterrupt:
            print("\nstream cancelled by user")
            raise

# Uncomment to try a live stream after setting ANTHROPIC_API_KEY.
# final_message = stream_completion("Draft a concise checklist for migrating prompts to Claude Opus 4.7.", effort="xhigh")
# print(final_message.to_json())

## 7. Generate structured JSON with schema validation

Structured outputs are the safest way to get parseable JSON back from Opus 4.7. This cell validates the response with Pydantic and retries with a repair prompt if needed.

In [ ]:
class ReleaseSummary(BaseModel):
    model_name: str
    best_fit: str
    effort: str = Field(description="Recommended effort level")
    caveats: list[str]


def extract_release_summary(text: str) -> ReleaseSummary:
    return ReleaseSummary.model_validate_json(text)


schema = ReleaseSummary.model_json_schema()
summary_result = call_claude(
    "Extract a compact JSON summary for Claude Opus 4.7 with keys model_name, best_fit, effort, and caveats.",
    max_tokens=600,
    effort="high",
    output_schema=schema,
)

try:
    summary = extract_release_summary(summary_result["text"])
except ValidationError:
    repair_result = call_claude(
        "Return only valid JSON that matches this schema:\n"
        f"{json.dumps(schema, indent=2)}\n\n"
        f"Bad output to repair:\n{summary_result['text']}",
        max_tokens=600,
        effort="high",
        output_schema=schema,
    )
    summary = extract_release_summary(repair_result["text"])

print(summary.model_dump())

## 8. Use tool calling with local Python functions

The SDK can run local Python tools for you. This is the cleanest way to build a small agent loop without inventing a separate framework.

In [ ]:
_ALLOWED_BINOPS = {
    ast.Add: lambda left, right: left + right,
    ast.Sub: lambda left, right: left - right,
    ast.Mult: lambda left, right: left * right,
    ast.Div: lambda left, right: left / right,
    ast.Mod: lambda left, right: left % right,
    ast.Pow: lambda left, right: left**right,
}
_ALLOWED_UNARYOPS = {
    ast.UAdd: lambda value: value,
    ast.USub: lambda value: -value,
}


def _safe_eval(node: ast.AST) -> float:
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINOPS:
        return _ALLOWED_BINOPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UNARYOPS:
        return _ALLOWED_UNARYOPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Only simple arithmetic expressions are allowed.")


@beta_tool
def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression."""
    value = _safe_eval(ast.parse(expression, mode="eval"))
    return json.dumps({"expression": expression, "result": value})


runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=800,
    tools=[calculator],
    messages=[
        {
            "role": "user",
            "content": "What is (18 + 6) * 3, and why is this useful when validating an agent tool loop?",
        }
    ],
)

for message in runner:
    print(message)

## 9. Build a long-context Q&A recipe

Opus 4.7 has a 1M token context window, but chunking still helps when you want better grounding, lower latency, and easier retrieval from a large document set.

In [ ]:
def chunk_text(text: str, *, chunk_size: int = 1200) -> list[str]:
    return [text[index : index + chunk_size] for index in range(0, len(text), chunk_size)]


documents = {
    "launch_note": (
        "Claude Opus 4.7 is generally available. It improves long-running coding tasks, "
        "uses effort levels for control, and supports 1M context."
    ),
    "migration_note": (
        "For Opus 4.7, remove non-default temperature, top_p, and top_k values. "
        "Use thinking={type: adaptive} and effort to tune reasoning depth."
    ),
}

annotated_chunks: list[str] = []
for source_name, source_text in documents.items():
    for index, chunk in enumerate(chunk_text(source_text), start=1):
        annotated_chunks.append(f"[{source_name}#{index}] {chunk}")

question = "What should a team change first when migrating prompt-heavy code to Opus 4.7?"
context_block = "\n\n".join(annotated_chunks)
prompt = f"Answer the question using only the sources below. Cite each claim with source labels in brackets.\n\nSources:\n{context_block}\n\nQuestion: {question}"

context_count = client.messages.count_tokens(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
)
print({"input_tokens": context_count.input_tokens})

qa_result = call_claude(prompt, max_tokens=500, effort="high", use_adaptive_thinking=True)
print(qa_result["text"])

## 10. Add retries, timeouts, and error handling

The SDK already retries transient failures, but agentic workloads often benefit from an explicit outer retry policy and tighter timeouts.

In [ ]:
def resilient_call(prompt: str, *, timeout_seconds: float = 30.0, max_retries: int = 0) -> Any:
    return client.with_options(timeout=timeout_seconds, max_retries=max_retries).messages.create(
        model=MODEL,
        max_tokens=800,
        messages=[{"role": "user", "content": prompt}],
        output_config={"effort": "high"},
    )


@retry(
    reraise=True,
    stop=stop_after_attempt(4),
    wait=wait_exponential_jitter(initial=1, max=8),
    retry=retry_if_exception_type(
        (
            anthropic.APIConnectionError,
            anthropic.APITimeoutError,
            anthropic.RateLimitError,
            anthropic.InternalServerError,
        )
    ),
)
def retrying_call(prompt: str) -> Any:
    return resilient_call(prompt, timeout_seconds=25.0, max_retries=0)


try:
    retry_response = retrying_call("Write one sentence about why Opus 4.7 is useful for long-horizon coding.")
    print(extract_text(retry_response))
    print({"request_id": retry_response._request_id, "usage": retry_response.usage.model_dump()})
except anthropic.APIStatusError as error:
    print({"status_code": error.status_code, "message": str(error)})

## 11. Track latency and token usage

Measure latency and token counts together so you can see the cost/performance trade-off of different prompts and effort levels.

In [ ]:
records: list[dict[str, Any]] = []
benchmarks = [
    ("short", "Give a short migration note for Opus 4.7.", "low"),
    ("standard", "Explain the main builder benefit of Opus 4.7.", "high"),
    ("agentic", "Describe how you would use Opus 4.7 for a long-running coding task.", "xhigh"),
]

for label, prompt_text, effort in benchmarks:
    start = time.perf_counter()
    response = call_claude(prompt_text, max_tokens=500, effort=effort, use_adaptive_thinking=(effort != "low"))
    elapsed = time.perf_counter() - start
    records.append(
        {
            "label": label,
            "effort": effort,
            "latency_s": round(elapsed, 2),
            "input_tokens": response["usage"].input_tokens,
            "output_tokens": response["usage"].output_tokens,
            "request_id": response["request_id"],
        }
    )

latency_df = pd.DataFrame(records)
latency_df["tokens_total"] = latency_df["input_tokens"] + latency_df["output_tokens"]
print(latency_df.sort_values("latency_s").to_string(index=False))

console = Console()
table = Table(title="Opus 4.7 latency and token snapshot")
for column in ["label", "effort", "latency_s", "input_tokens", "output_tokens", "tokens_total"]:
    table.add_column(column)

for row in latency_df.itertuples(index=False):
    table.add_row(
        row.label,
        row.effort,
        str(row.latency_s),
        str(row.input_tokens),
        str(row.output_tokens),
        str(row.tokens_total),
    )

console.print(table)

## 12. Batch cookbook generation from a CSV input

Batch jobs are the right shape when you want to turn a prompt list into a repeatable cookbook generator or a bulk review workflow.

In [ ]:
output_dir = Path("opus-4-7-batch-output")
output_dir.mkdir(exist_ok=True)

csv_path = output_dir / "recipe_prompts.csv"
if not csv_path.exists():
    pd.DataFrame(
        [
            {"custom_id": "recipe-1", "prompt": "Summarize the biggest migration risk in Opus 4.7."},
            {"custom_id": "recipe-2", "prompt": "List three cases where xhigh is worth the extra spend."},
        ]
    ).to_csv(csv_path, index=False)

batch_source = pd.read_csv(csv_path)
requests = [
    {
        "custom_id": row.custom_id,
        "params": {
            "model": MODEL,
            "max_tokens": 600,
            "messages": [{"role": "user", "content": row.prompt}],
            "output_config": {"effort": "high"},
        },
    }
    for row in batch_source.itertuples(index=False)
]

batch = client.messages.batches.create(requests=requests)
print({"batch_id": batch.id, "status": batch.processing_status})

# Poll until the batch finishes before pulling results.
# This is intentionally simple so you can swap in your own job scheduler later.
while batch.processing_status != "ended":
    time.sleep(10)
    batch = client.messages.batches.retrieve(batch.id)
    print({"batch_id": batch.id, "status": batch.processing_status})

results = []
for entry in client.messages.batches.results(batch.id):
    if entry.result.type == "succeeded":
        text = extract_text(entry.result.message)
        results.append({"custom_id": entry.custom_id, "text": text})

results_df = pd.DataFrame(results)
results_df.to_json(output_dir / "batch_results.json", orient="records", indent=2)
results_df.to_csv(output_dir / "batch_results.csv", index=False)
print(results_df.to_string(index=False))

### Practical notes

- Opus 4.7 is strongest when you give it `high` or `xhigh` effort for coding and agentic work.
- High-resolution image support reaches 2576 pixels on the long edge, so keep full fidelity only when you actually need it.
- If a workflow runs for a long time, consider task budgets so the model can self-scope the run instead of drifting.
- Keep `temperature`, `top_p`, and `top_k` out of Opus 4.7 requests unless you are using the default values.